# Chapter 15 - Deep Learning on Unseen Data: Introducing Federated Learning

## Summary
Normally all training data is gathered on one server. But the most valuable data (messages, medical records, photos) is often **private**. Federated learning brings the *model to the data* instead of the data to the model.
- **Privacy problem:** collecting data centrally exposes users and may be illegal or impractical.
- **Federated learning (FL):** each participant (client) trains the model *locally* on its own data; only **model updates/weights** are sent to a server, which **averages** them (FedAvg) and sends the improved model back.
- **Federated averaging loop:** (1) server sends model to clients, (2) clients train locally for a few steps, (3) clients return weights, (4) server averages -> new global model.
- **Attacks still exist:** weight updates can leak information about the data (e.g. a gradient can reveal training examples).
- **Secure aggregation / additive secret sharing:** clients split their update into random *shares* so the server only sees the sum, never an individual update.
- **Homomorphic encryption:** allows computing (adding) on encrypted values.
- **Differential privacy:** add calibrated random noise so no single person's data noticeably changes the output.
- (The book builds this with the PySyft library; here it is simulated from scratch with NumPy so it runs anywhere.)

## Theory
- **FedAvg:** `w_global = sum_k (n_k / n) * w_k` - a weighted average of client models, weighted by their data amount `n_k`.
- **Non-IID data:** clients rarely have identically distributed data; FL tolerates this but converges slower.
- **Additive secret sharing:** to hide a value `x` from the server, produce shares `s_1 ... s_m` where `s_m = x - (s_1 + ... + s_{m-1})` and the others are random. Any incomplete set of shares looks random, but all shares sum to `x`.
- **Homomorphic property:** the (encrypted/shared) sum of values equals the encrypted sum - so the server can add updates without reading them.
- **Differential privacy:** mechanism `M` is epsilon-DP if adding/removing one record changes output probabilities by at most `e^epsilon`. In practice: clip each update to norm `C`, add Gaussian noise `N(0, sigma^2 C^2)`.
- **Trade-off:** more privacy (noise) -> lower accuracy.

In [1]:
# If packages are missing in this kernel, uncomment:
# %pip install numpy scikit-learn matplotlib
import numpy as np

In [2]:
from sklearn.datasets import load_digits
digits = load_digits()
X = digits.data / 16.0; y = digits.target
rng = np.random.RandomState(0)
perm = rng.permutation(len(X)); X, y = X[perm], y[perm]
X_test, y_test = X[1400:], y[1400:]
X_tr, y_tr = X[:1400], y[:1400]
n_clients = 4
clients = [(X_tr[i::n_clients], y_tr[i::n_clients]) for i in range(n_clients)]   # each client keeps own data
print([len(c[0]) for c in clients], "samples per client | test:", len(X_test))

def softmax(z):
    e = np.exp(z - z.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)
def init(): return np.zeros((64, 10))
def local_train(W, Xc, yc, epochs=5, lr=0.5, batch=32):
    W = W.copy()
    for _ in range(epochs):
        order = rng.permutation(len(Xc))
        for s in range(0, len(Xc), batch):
            b = order[s:s+batch]
            p = softmax(Xc[b].dot(W)); p[np.arange(len(b)), yc[b]] -= 1
            W -= lr * Xc[b].T.dot(p) / len(b)
    return W
def accuracy(W): return np.mean(softmax(X_test.dot(W)).argmax(1) == y_test)

[350, 350, 350, 350] samples per client | test: 397


### 1) Baselines: centralized training vs. a single client alone

In [3]:
W_central = local_train(init(), X_tr, y_tr, epochs=20)
W_single  = local_train(init(), *clients[0], epochs=20)
print(f"centralized (all data pooled): {accuracy(W_central):.3f}")
print(f"one client only (350 samples): {accuracy(W_single):.3f}")

centralized (all data pooled): 0.970
one client only (350 samples): 0.942


### 2) Federated averaging (data never leaves the clients)

In [4]:
W_global = init()
for rnd in range(1, 11):
    local_models = [local_train(W_global, Xc, yc, epochs=2) for Xc, yc in clients]   # local training
    sizes = np.array([len(c[0]) for c in clients])
    W_global = sum(w * (n / sizes.sum()) for w, n in zip(local_models, sizes))        # FedAvg
    if rnd % 2 == 0 or rnd == 1: print(f"round {rnd:2d}  global test accuracy {accuracy(W_global):.3f}")

round  1  global test accuracy 0.914
round  2  global test accuracy 0.917
round  4  global test accuracy 0.932
round  6  global test accuracy 0.932
round  8  global test accuracy 0.940
round 10  global test accuracy 0.950


Federated training approaches the centralized accuracy **without ever pooling the raw data**.

### 3) Secure aggregation with additive secret sharing
Each client splits its update into random shares; the server only ever sees *sums of shares*.

In [5]:
def make_shares(x, n_shares):
    shares = [rng.randn(*x.shape) * 10 for _ in range(n_shares - 1)]
    shares.append(x - sum(shares))
    return shares

updates = [local_train(W_global, Xc, yc, epochs=2) - W_global for Xc, yc in clients]
shares = [make_shares(u, n_clients) for u in updates]             # client k -> shares for each client
# party j receives share j from every client and only publishes the sum of what it received
partial_sums = [sum(shares[k][j] for k in range(n_clients)) for j in range(n_clients)]
secure_sum = sum(partial_sums)

print("share looks like noise (std):", shares[0][0].std().round(2), "| true update std:", updates[0].std().round(4))
print("secure sum == true sum of updates?", np.allclose(secure_sum, sum(updates)))

share looks like noise (std): 10.21 | true update std: 0.0198
secure sum == true sum of updates? True


### 4) Differential privacy: clip updates and add Gaussian noise

In [6]:
def dp_federated(noise_mult, clip=1.0, rounds=10):
    W = init()
    for _ in range(rounds):
        deltas = []
        for Xc, yc in clients:
            d = local_train(W, Xc, yc, epochs=2) - W
            d *= min(1.0, clip / (np.linalg.norm(d) + 1e-12))           # clip update norm
            deltas.append(d)
        avg = sum(deltas) / len(deltas)
        avg += rng.randn(*avg.shape) * noise_mult * clip / len(deltas)  # Gaussian noise
        W = W + avg
    return accuracy(W)
for nm in [0.0, 0.1, 0.5, 1.0, 2.0]:
    print(f"noise multiplier {nm:3.1f} -> accuracy {dp_federated(nm):.3f}")

noise multiplier 0.0 -> accuracy 0.940
noise multiplier 0.1 -> accuracy 0.937
noise multiplier 0.5 -> accuracy 0.919
noise multiplier 1.0 -> accuracy 0.617
noise multiplier 2.0 -> accuracy 0.287


More noise gives stronger privacy but lower accuracy - the **privacy / utility trade-off**.

## Takeaways
- Federated learning keeps raw data on the owner's device; only model updates travel.
- FedAvg approximates centralized training; secret sharing / homomorphic encryption hide individual updates; differential privacy bounds what any update can reveal.
- This is the book's closing message: deep learning can be done **responsibly**, and the toolbox (autograd, layers, optimizers) you built carries over to real frameworks like PyTorch.